In [13]:
import torch
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torchvision.models import vgg19, VGG19_Weights
import torch.nn as nn
import torch.optim as optim

In [2]:
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225])
])
train_data = CIFAR10(root='./data',train=True,download=True,transform=transform)
test_data = CIFAR10(root='./data',train=False,download=True,transform=transform)

In [3]:
train_loader = DataLoader(train_data,batch_size=32,shuffle=True)
test_loader = DataLoader(test_data,batch_size=32,shuffle=False)

In [5]:
model = vgg19(weights=VGG19_Weights.DEFAULT)

In [9]:
model.classifier[6] = nn.Linear(in_features=4096, out_features=10, bias=True)

In [14]:
model

In [25]:
for param in model.parameters():
    param.requires_grad = False
    
num_classes = len(train_loader.dataset.classes)
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_features, num_classes)

In [26]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [15]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

In [17]:
device = torch.device('mps' if torch.mps.is_available else "cpu")
print(device)
model = model.to(device)

mps


In [ ]:
from tqdm import tqdm

def train_loop(model, train_loader, criterion, optimizer):
  model.train()

  running_loss = 0
  correct = 0
  total = 0

  for images, labels in tqdm(train_loader):
    images = images.to(device)
    labels = labels.to(device)

    # Prediction
    logits = model(images)

    # Compute the loss
    loss = criterion(logits,labels)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # Weight update
    optimizer.step()

    running_loss += loss.item()


    predictions = torch.argmax(logits, dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

  epoch_loss = running_loss / len(train_loader)
  epoch_acc = correct / total

  print(f"Train Loss : {epoch_loss:.4f} | Train Acc : {epoch_acc:.4f}")


In [31]:
from tqdm import tqdm
def train_loop2(model,train_loader,criterion,optimizer):
    model.train()
    running_loss, correct, total = 0,0,0
    for image,label in tqdm(train_loader):
        image  = image.to(device)
        label = label.to(device)
        logits = model(image)
        loss = criterion(logits,label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()
        predictions = torch.argmax(logits,dim=1)
        correct += (predictions==label).sum().item()
        total += label.size(0)

    epoch_loss = running_loss/len(train_loader)
    epoc_acc = correct/total

    print(f"Tran Loss : {epoch_loss:.4f} | Train Accuracy : {epoc_acc:.4f}")

In [32]:
model = model.to(device)
for epoch in range(2):
    print(f"Epoch : {epoch+1}")
    train_loop2(model,train_loader,criterion,optimizer)

Epoch : 1


100%|██████████████████████████████████████| 1563/1563 [13:48<00:00,  1.89it/s]


Tran Loss : 2.4155 | Train Accuracy : 0.0912
Epoch : 2


100%|██████████████████████████████████████| 1563/1563 [14:51<00:00,  1.75it/s]

Tran Loss : 2.4149 | Train Accuracy : 0.0966
